# Taxonomy Clade Examples

This notebook shows how to use `CBBIO.Taxonomy` with local NCBI taxdump files (`nodes.dmp`, `names.dmp`) to compute:

- Lowest common ancestor (LCA) ID/rank/clade
- Whether two taxa share a clade at a target rank
- `depth(LCA)/max(depth(a), depth(b))`
- Wu-Palmer taxonomy similarity


## Download required NCBI taxdump files

This notebook expects `nodes.dmp` and `names.dmp` under `../taxdump` (relative to `BioData/notebooks/`).

Run these commands once from the `BioData/` directory:

```bash
mkdir -p taxdump
curl -L https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz -o taxdump/taxdump.tar.gz
tar -xzf taxdump/taxdump.tar.gz -C taxdump nodes.dmp names.dmp
```

After extraction, these files must exist:
- `BioData/taxdump/nodes.dmp`
- `BioData/taxdump/names.dmp`


In [8]:
import importlib
import sys
from pathlib import Path

def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for base in candidates:
        base = base.resolve()
        if (base / 'CBBIO').exists() and (base / 'README.md').exists():
            return base
    raise FileNotFoundError('Could not locate BioData repo root containing CBBIO/.')

def resolve_taxdump_dir(taxdump_path_str: str, repo_root: Path) -> Path:
    raw = Path(taxdump_path_str)
    candidates = [
        raw,
        Path.cwd() / raw,
        repo_root / raw,
        repo_root / 'taxdump',
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    for base in candidates:
        base = base.resolve()
        if (base / 'nodes.dmp').exists() and (base / 'names.dmp').exists():
            return base
    raise FileNotFoundError(
        'Could not find nodes.dmp and names.dmp. Expected under BioData/taxdump or provide a valid path.'
    )

repo_root = resolve_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Force local import even if CBBIO was previously imported from site-packages.
for mod_name in list(sys.modules):
    if mod_name == 'CBBIO' or mod_name.startswith('CBBIO.'):
        del sys.modules[mod_name]

taxonomy_mod = importlib.import_module('CBBIO.Taxonomy')
load_taxonomy = taxonomy_mod.load_taxonomy

taxdump_dir = resolve_taxdump_dir("../taxdump", repo_root)
print('Using repo root:', repo_root)
print('Using taxdump dir:', taxdump_dir)
print('Imported taxonomy module from:', Path(taxonomy_mod.__file__).resolve())

required = ['lowest_common_ancestor_rank', 'lowest_common_ancestor_clade', 'shares_clade_at_rank', 'wu_palmer_similarity']
for name in required:
    assert hasattr(taxonomy_mod.TaxonomyOntology, name), f'Missing method in imported TaxonomyOntology: {name}'


Using repo root: /home/icases/evaluator/BioData
Using taxdump dir: /home/icases/evaluator/BioData/taxdump
Imported taxonomy module from: /home/icases/evaluator/BioData/CBBIO/Taxonomy.py


In [9]:
tax = load_taxonomy(str(taxdump_dir))
print(f'Loaded taxonomy IDs: {len(tax.taxon_ids):,}')


Loaded taxonomy IDs: 2,730,551


## Prepare information content background

In [10]:
from CBBIO.BioData import BioDataClient

with BioDataClient() as client:
    rows = client.query_all(
        """
        SELECT id, taxonomy_id
        FROM protein
        WHERE taxonomy_id IS NOT NULL;
        """
    )

annotations = {
    str(row['id']): {str(row['taxonomy_id'])}
    for row in rows
    if row.get('id') and row.get('taxonomy_id')
}

if not annotations:
    raise RuntimeError('No taxonomy annotations were retrieved from DB. Check DB connection/config/data.')

tax.prepare_taxon_counts(annotations, mode='observed')

## Pairwise comparison

Set two taxonomy IDs to compare.


In [11]:
# Example pair (edit as needed):
# 9606 = human
# 9796 = horse
# This intentionally gives different depths for A vs B.
taxon_id_A = "9606"
taxon_id_B = "9796"

taxon_A = tax.taxon(taxon_id_A)
taxon_B = tax.taxon(taxon_id_B)

lca_id = tax.lowest_common_ancestor(taxon_id_A, taxon_id_B)
lca_rank = tax.lowest_common_ancestor_rank(taxon_id_A, taxon_id_B)
lca_clade = tax.lowest_common_ancestor_clade(taxon_id_A, taxon_id_B)

result = {
    "taxon_A": taxon_A,
    "taxon_B": taxon_B,
    "depth_A": taxon_A['depth'],
    "depth_B": taxon_B['depth'],
    "depths_equal": taxon_A['depth'] == taxon_B['depth'],
    "lca_id": lca_id,
    "lca_rank": lca_rank,
    "lca_clade": lca_clade,
    "same_genus": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "genus"),
    "same_family": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "family"),
    "same_order": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "order"),
    "normalized_lca_depth": tax.normalized_lca_depth(taxon_id_A, taxon_id_B),
    "wu_palmer_similarity": tax.wu_palmer_similarity(taxon_id_A, taxon_id_B),
     'ic_taxon_A': tax.information_content(taxon_id_A, mode='observed'),
    'ic_taxon_B': tax.information_content(taxon_id_B, mode='observed'),
    'ic_lca': tax.information_content(lca_id, mode='observed') if lca_id else None,
}

result


   

{'taxon_A': {'id': '9606',
  'name': 'Homo sapiens',
  'rank': 'species',
  'parent_id': '9605',
  'depth': 31},
 'taxon_B': {'id': '9796',
  'name': 'Equus caballus',
  'rank': 'species',
  'parent_id': '9789',
  'depth': 26},
 'depth_A': 31,
 'depth_B': 26,
 'depths_equal': False,
 'lca_id': '1437010',
 'lca_rank': 'clade',
 'lca_clade': {'id': '1437010',
  'name': 'Boreoeutheria',
  'rank': 'clade',
  'depth': 21},
 'same_genus': False,
 'same_family': False,
 'same_order': False,
 'normalized_lca_depth': 0.6774193548387096,
 'wu_palmer_similarity': 0.7368421052631579,
 'ic_taxon_A': 1.7081974688390695,
 'ic_taxon_B': 8.865413607452789,
 'ic_lca': 0.9523803950566165}

In [12]:
# Example pair (edit as needed):
# 9606 = human
# 7955 = zebra fish
# This intentionally gives different depths for A vs B.
taxon_id_A = "9606"
taxon_id_B = "7955"

taxon_A = tax.taxon(taxon_id_A)
taxon_B = tax.taxon(taxon_id_B)

lca_id = tax.lowest_common_ancestor(taxon_id_A, taxon_id_B)
lca_rank = tax.lowest_common_ancestor_rank(taxon_id_A, taxon_id_B)
lca_clade = tax.lowest_common_ancestor_clade(taxon_id_A, taxon_id_B)

result = {
    "taxon_A": taxon_A,
    "taxon_B": taxon_B,
    "depth_A": taxon_A['depth'],
    "depth_B": taxon_B['depth'],
    "depths_equal": taxon_A['depth'] == taxon_B['depth'],
    "lca_id": lca_id,
    "lca_rank": lca_rank,
    "lca_clade": lca_clade,
    "same_genus": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "genus"),
    "same_family": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "family"),
    "same_order": tax.shares_clade_at_rank(taxon_id_A, taxon_id_B, "order"),
    "normalized_lca_depth": tax.normalized_lca_depth(taxon_id_A, taxon_id_B),
    "wu_palmer_similarity": tax.wu_palmer_similarity(taxon_id_A, taxon_id_B),
     'ic_taxon_A': tax.information_content(taxon_id_A, mode='observed'),
    'ic_taxon_B': tax.information_content(taxon_id_B, mode='observed'),
    'ic_lca': tax.information_content(lca_id, mode='observed') if lca_id else None,
}

result



{'taxon_A': {'id': '9606',
  'name': 'Homo sapiens',
  'rank': 'species',
  'parent_id': '9605',
  'depth': 31},
 'taxon_B': {'id': '7955',
  'name': 'Danio rerio',
  'rank': 'species',
  'parent_id': '7954',
  'depth': 29},
 'depth_A': 31,
 'depth_B': 29,
 'depths_equal': False,
 'lca_id': '117571',
 'lca_rank': 'clade',
 'lca_clade': {'id': '117571',
  'name': 'Euteleostomi',
  'rank': 'clade',
  'depth': 13},
 'same_genus': False,
 'same_family': False,
 'same_order': False,
 'normalized_lca_depth': 0.41935483870967744,
 'wu_palmer_similarity': 0.43333333333333335,
 'ic_taxon_A': 1.7081974688390695,
 'ic_taxon_B': 2.6793193019290995,
 'ic_lca': 0.7211376660767072}